# Grokking em Redes Neurais: O Caso do Perceptron Multicamadas (MLP)

Este notebook demonstra o fenômeno de **grokking** em um Perceptron Multicamadas (MLP) simples treinado na tarefa de **soma modular**:

$$a + b \pmod p$$

onde $p = 67$ é um número primo. O grokking ocorre quando a rede neural atinge acurácia de 100% no conjunto de treinamento rapidamente (fase de memorização), mas precisa de milhares de épocas adicionais para que a acurácia de validação salte repentinamente de 0% para 100% (fase de generalização).

Conectamos esse comportamento à física de transições de fase de primeira ordem sob a ação do termo de regularização *Weight Decay* (decaimento de pesos).

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Configurar sementes para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando o dispositivo: {device}")

### 1. Definição do Modelo MLP

O modelo possui:
1. Camada de embedding compartilhado de dimensão `embed_dim` (64).
2. Camada oculta linear que projeta a concatenação dos operandos para `hidden_dim` (128).
3. Ativação não-linear GELU.
4. Camada de saída linear projetando para o vocabulário de classes $p$.

In [ ]:
class ModularMLP(nn.Module):
    def __init__(self, p, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(p, embed_dim)
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, p)
        
    def forward(self, x):
        # x shape: (B, 2) onde cada exemplo é [a, b]
        e = self.embed(x) # (B, 2, embed_dim)
        e = e.view(e.shape[0], -1) # (B, 2 * embed_dim)
        h = self.act(self.fc1(e))
        logits = self.fc2(h)
        return logits

### 2. Geração e Divisão dos Dados

Geramos todas as $p^2$ combinações possíveis. Para ver o grokking, dividimos em uma fração pequena de treino (35%) e o restante para validação (65%).

In [ ]:
p = 67
train_fraction = 0.35

# Criar todas as possíveis somas modular
x_data = []
y_data = []
for a in range(p):
    for b in range(p):
        x_data.append([a, b])
        y_data.append((a + b) % p)

x_data = torch.tensor(x_data, dtype=torch.long)
y_data = torch.tensor(y_data, dtype=torch.long)

# Divisão de treino e teste baseada em permutação
num_samples = len(x_data)
indices = np.random.permutation(num_samples)
split_idx = int(num_samples * train_fraction)

train_idx = indices[:split_idx]
val_idx = indices[split_idx:]

x_train, y_train = x_data[train_idx], y_data[train_idx]
x_val, y_val = x_data[val_idx], y_data[val_idx]

print(f"Quantidade total de dados: {num_samples}")
print(f"Tamanho do conjunto de Treino: {len(x_train)} ({train_fraction*100:.1f}%)")
print(f"Tamanho do conjunto de Validação: {len(x_val)}")

### 3. Loop de Treinamento com Weight Decay

Treinamos com AdamW, taxa de aprendizado $10^{-3}$ e decaimento de pesos (Weight Decay) ajustado para $1.0$. Registramos a perda, a acurácia e a norma dos pesos sinápticos ao longo das épocas.

In [ ]:
model = ModularMLP(p=p, embed_dim=64, hidden_dim=128).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1.0)
criterion = nn.CrossEntropyLoss()

epochs = 12000
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'weight_norm': []
}

x_train, y_train = x_train.to(device), y_train.to(device)
x_val, y_val = x_val.to(device), y_val.to(device)

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    
    # Avaliar métricas
    model.eval()
    with torch.no_grad():
        train_preds = logits.argmax(dim=-1)
        train_acc = (train_preds == y_train).float().mean().item()
        
        val_logits = model(x_val)
        val_loss = criterion(val_logits, y_val).item()
        val_preds = val_logits.argmax(dim=-1)
        val_acc = (val_preds == y_val).float().mean().item()
        
        # Calcular norma L2 total dos pesos
        w_norm = sum(p.pow(2).sum() for p in model.parameters()).sqrt().item()
        
    history['train_loss'].append(loss.item())
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['weight_norm'].append(w_norm)
    
    if (epoch + 1) % 1000 == 0 or epoch == 0:
        print(f"Época {epoch+1:5d}/{epochs} | "
              f"Perda Treino: {loss.item():.4f} | Acurácia Treino: {train_acc*100:6.2f}% | "
              f"Perda Val: {val_loss:.4f} | Acurácia Val: {val_acc*100:6.2f}% | "
              f"Norma dos Pesos: {w_norm:.2f}")
        
        # Parada antecipada se tiver generalizado de forma estável
        if train_acc > 0.999 and val_acc > 0.99:
            if len(history['val_acc']) > 10 and all(a > 0.98 for a in history['val_acc'][-5:]):
                print(f"\nGrokking alcançado com sucesso na época {epoch+1}!")
                break

### 4. Visualização das Curvas de Aprendizado

Abaixo plotamos as curvas de perda e acurácia. Observe a transição brusca da acurácia de validação de ~1.5% para 100%, muito após a acurácia de treino atingir 100%.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(6.5, 9.5), sharex=True)

# Perda
ax1.plot(history['train_loss'], label='Treino', color='blue', alpha=0.7)
ax1.plot(history['val_loss'], label='Validação', color='red', alpha=0.7)
ax1.set_yscale('log')
ax1.set_title('Perda (Loss) vs Épocas')
ax1.set_ylabel('Perda (escala log)')
ax1.legend()
ax1.grid(True, which='both', ls='--')

# Acurácia
ax2.plot(history['train_acc'], label='Treino', color='blue', alpha=0.7)
ax2.plot(history['val_acc'], label='Validação', color='red', alpha=0.7)
ax2.set_title('Acurácia vs Épocas')
ax2.set_ylabel('Acurácia (%)')
ax2.legend()
ax2.grid(True, ls='--')

# Norma dos Pesos
ax3.plot(history['weight_norm'], label='Norma L2', color='purple', alpha=0.7)
ax3.set_title('Norma dos Pesos vs Épocas')
ax3.set_xlabel('Épocas')
ax3.set_ylabel('Norma L2')
ax3.legend()
ax3.grid(True, ls='--')

plt.suptitle('Grokking como Transição de Fase Dinâmica em MLP', fontsize=12, y=0.99)
plt.tight_layout()
plt.show()

### 5. Análise Geométrica dos Embeddings Aprendidos

Na fase de generalização, a rede aprende a simetria cíclica discreta $C_p$ projetando os índices discretos $\{0, \dots, p-1\}$ em círculos bidimensionais correspondentes a frequências de Fourier. 

Extraímos a matriz de pesos de embedding $W_E \in \mathbb{R}^{p \times d_{\text{embed}}}$ e aplicamos PCA para projetar em 2 dimensões.

In [ ]:
# Extrair os pesos de embedding
embeddings = model.embed.weight.data.cpu().numpy()

# Aplicar PCA para reduzir de 64 para 2 dimensões
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Plotar os embeddings projetados no plano
plt.figure(figsize=(8, 8))
colors = plt.cm.hsv(np.linspace(0, 1, p))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=range(p), cmap='hsv', s=100, edgecolors='k')

# Numerar cada ponto correspondente ao inteiro modular
for i in range(p):
    plt.annotate(str(i), (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                 textcoords="offset points", xytext=(0,5), ha='center', fontsize=9, fontweight='bold')

plt.title('Representação Circular dos Inteiros Modulares (Embedding Pós-Grokking)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True, ls='--')
plt.axis('equal')
plt.show()

### 6. Análise Espectral de Fourier nos Embeddings

Para comprovar matematicamente que a rede neural aprendeu circuitos trigonométricos periódicos, calculamos a **Transformada Discreta de Fourier (DFT)** 1D ao longo da dimensão do vocabulário na matriz de pesos de embedding $W_E \in \mathbb{R}^{p \times d_{\text{embed}}}$.

Para cada recurso (coluna) do embedding, calculamos sua FFT e extraímos o espectro de potência médio normalizado. Modelos generalizadores exibirão picos nítidos em números de onda ($k$) inteiros específicos.

In [ ]:
# Calcular a DFT ao longo do vocabulário (eixo 0)
fft_weights = np.fft.fft(embeddings, axis=0)
power_spectrum = np.abs(fft_weights) ** 2

# Considerar apenas frequências positivas devido à simetria da DFT
half_p = p // 2 + 1
power_spectrum = power_spectrum[:half_p, :]
frequencies = np.arange(half_p)

# Espectro médio sobre as d_embed = 64 dimensões do embedding
mean_power = np.mean(power_spectrum, axis=1)
mean_power /= np.sum(mean_power) # Normalizar

# Plotar usando stem plot
plt.figure(figsize=(9, 4.5))
plt.stem(frequencies, mean_power, linefmt='b-', markerfmt='bo', basefmt='r-')
plt.title('Espectro de Potência Médio Normalizado dos Embeddings (Pós-Grokking)')
plt.xlabel('Frequência (número de onda k)')
plt.ylabel('Potência Normalizada')
plt.grid(True, ls='--', alpha=0.5)
plt.xlim(-1, half_p)
plt.show()

### 7. Contraste: O que acontece SEM Weight Decay?

Se removermos a regularização de decaimento de pesos (`weight_decay = 0.0`), a rede atinge $100\%$ de treino memorizando os dados, mas nunca transiciona para a generalização. Isso prova que o decaimento de pesos atua como um campo térmico que força o resfriamento e reorganização do sistema.

In [ ]:
print("Treinando SEM Weight Decay...")
model_no_wd = ModularMLP(p=p, embed_dim=64, hidden_dim=128).to(device)
optimizer_no_wd = optim.AdamW(model_no_wd.parameters(), lr=1e-3, weight_decay=0.0)

epochs_test = 4000
history_no_wd = {'train_acc': [], 'val_acc': []}

for epoch in range(epochs_test):
    model_no_wd.train()
    optimizer_no_wd.zero_grad()
    logits = model_no_wd(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer_no_wd.step()
    
    model_no_wd.eval()
    with torch.no_grad():
        train_acc = (logits.argmax(dim=-1) == y_train).float().mean().item()
        val_logits = model_no_wd(x_val)
        val_acc = (val_logits.argmax(dim=-1) == y_val).float().mean().item()
        
    history_no_wd['train_acc'].append(train_acc)
    history_no_wd['val_acc'].append(val_acc)
    
    if (epoch + 1) % 1000 == 0:
        print(f"Época {epoch+1:4d} | Acurácia Treino: {train_acc*100:.2f}% | Acurácia Val: {val_acc*100:.2f}%")

# Plotar comparação
plt.figure(figsize=(10, 5))
plt.plot(history_no_wd['train_acc'], label='Treino (Sem WD)', color='blue', ls='--')
plt.plot(history_no_wd['val_acc'], label='Validação (Sem WD)', color='red', ls='--')
plt.title('Acurácia ao longo do tempo (Sem Weight Decay)')
plt.xlabel('Épocas')
plt.ylabel('Acurácia (%)')
plt.legend()
plt.grid(True, ls='--')
plt.show()